In [5]:
import importlib
import torch
import model_def
import numpy as np
importlib.reload(model_def)
from torch import nn
from pathlib import Path
from model_def import get_model
from test_mnist import download, load_images, load_labels
from emnist import extract_training_samples

In [6]:
def load_mnist_train(root="./data"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    image_file = "train-images-idx3-ubyte.gz"
    label_file = "train-labels-idx1-ubyte.gz"
    image_path = root / image_file
    label_path = root / label_file

    mnist_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
    download(mnist_url + image_file, image_path)
    download(mnist_url + label_file, label_path)

    images = load_images(image_path)
    labels = load_labels(label_path)

    images = images[:, None, :, :]
    images = images.repeat(3, axis=1)

    return images, labels

In [7]:
def load_emnist_train():
    emnist_images, emnist_labels = extract_training_samples('digits')
    emnist_images = emnist_images.astype(np.float32) / 255.0
    emnist_images = emnist_images[:, None, :, :]
    emnist_images = emnist_images.repeat(3, axis=1)
    return emnist_images, emnist_labels

In [8]:
# emnist_images, emnist_labels = load_emnist_train()
# import matplotlib.pyplot as plt
# for i in [0, 1, 2, 3, 4]:
#     print(f"index {i}, label says: {emnist_labels[i]}")
#     plt.imshow(emnist_images[i, 0], cmap='gray')
#     plt.title(f"label: {emnist_labels[i]}")
#     plt.show()

In [9]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.0+cu130
True
NVIDIA GeForce RTX 4060


In [10]:

#### THIS TRAINING IS OUT OF DATA AND NEEDS A DIFF LOSS FUNCTION
def train():
    images, labels = load_mnist_train()
    x = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 64
    n = x.shape[0]

    for epoch in range(10):
        perm = torch.randperm(n)          # shuffle indices each epoch
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = x[idx], y[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train()

epoch 0, avg loss 1.5579
epoch 1, avg loss 1.4925
epoch 2, avg loss 1.4851
epoch 3, avg loss 1.4818


KeyboardInterrupt: 

In [11]:
import math
import torch.nn.functional as F

#let's add color
def random_tint(images):
    n = images.shape[0]
    tint = torch.rand(n,3,1,1)*0.8+0.2
    return images*tint

def random_fg_bg_tint(images):
    n = images.shape[0]
    fg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    bg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    mask = images[ :, 0:1, :, :]
    colored_images = mask*fg_color + (1-mask)*bg_color
    return colored_images

def random_rotate(images, max_angle=45):
    n = images.shape[0]

    angles_deg = (torch.rand(n) * 2 - 1) * max_angle
    angles_rad = angles_deg * math.pi / 180.0

    cos = torch.cos(angles_rad)
    sin = torch.sin(angles_rad)

    theta = torch.zeros(n, 2, 3)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin
    theta[:, 1, 0] = sin
    theta[:, 1, 1] = cos

    grid = F.affine_grid(theta, images.shape, align_corners=False)
    rotated = F.grid_sample(images, grid, align_corners=False)

    return rotated

def random_occlude(images, max_patches=2, max_patch_size=12):
    """
    Randomly occludes each image with 0-max_patches rectangular patches,
    filled with a random solid color.
    images: torch tensor, shape (N, 3, H, W)
    returns: torch tensor, same shape
    """
    n, c, h, w = images.shape
    occluded = images.clone()

    for i in range(n):
        num_patches = torch.randint(0, max_patches + 1, (1,)).item()
        for _ in range(num_patches):
            patch_h = torch.randint(4, max_patch_size + 1, (1,)).item()
            patch_w = torch.randint(4, max_patch_size + 1, (1,)).item()

            top = torch.randint(0, max(1, h - patch_h), (1,)).item()
            left = torch.randint(0, max(1, w - patch_w), (1,)).item()

            patch_color = torch.rand(c, 1, 1)  # random solid color, one per channel
            occluded[i, :, top:top+patch_h, left:left+patch_w] = patch_color

    return occluded

In [12]:
def train_with_color():
    images, labels = load_mnist_train()
    emnist_images, emnist_labels = load_emnist_train()
    images = np.concatenate([images, emnist_images])
    labels = np.concatenate([labels, emnist_labels])

    x = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.NLLLoss()

    batch_size = 64
    n = x.shape[0]
    for epoch in range(80):
        perm = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = x[idx], y[idx]

            # each augmentation applied independently, with its own probability
            if torch.rand(1).item() < 0.3:
                xb = random_tint(xb)
            if torch.rand(1).item() < 0.3:
                xb = random_fg_bg_tint(xb)
            if torch.rand(1).item() < 0.4:
                xb = random_rotate(xb)
            if torch.rand(1).item() < 0.4:
                xb = random_occlude(xb)

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(torch.log(out + 1e-8), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.to("cpu").state_dict(), "trained.pt")
    print("saved trained.pt")

train_with_color()

epoch 0, avg loss 0.5714
epoch 1, avg loss 0.3266
epoch 2, avg loss 0.2706
epoch 3, avg loss 0.2477
epoch 4, avg loss 0.2276
epoch 5, avg loss 0.2152
epoch 6, avg loss 0.2053
epoch 7, avg loss 0.1996
epoch 8, avg loss 0.1924
epoch 9, avg loss 0.1917
epoch 10, avg loss 0.1868
epoch 11, avg loss 0.1825
epoch 12, avg loss 0.1777
epoch 13, avg loss 0.1757
epoch 14, avg loss 0.1720
epoch 15, avg loss 0.1728
epoch 16, avg loss 0.1697
epoch 17, avg loss 0.1681
epoch 18, avg loss 0.1640
epoch 19, avg loss 0.1566
epoch 20, avg loss 0.1657
epoch 21, avg loss 0.1597
epoch 22, avg loss 0.1618
epoch 23, avg loss 0.1541
epoch 24, avg loss 0.1523
epoch 25, avg loss 0.1567
epoch 26, avg loss 0.1551
epoch 27, avg loss 0.1543
epoch 28, avg loss 0.1518
epoch 29, avg loss 0.1529
epoch 30, avg loss 0.1516
epoch 31, avg loss 0.1471
epoch 32, avg loss 0.1454
epoch 33, avg loss 0.1487
epoch 34, avg loss 0.1469
epoch 35, avg loss 0.1443
epoch 36, avg loss 0.1430
epoch 37, avg loss 0.1434
epoch 38, avg loss 0.1